# Chapter 5 — A Minimal Code-Executing Agent

Hands-on lab: build the backbone agent and solve three small tasks (a math
problem, a file transform, an API-free data task), per
`agent_code_execution_study_guide.md` Chapter 5's hands-on direction.

This is the first chapter in the guide that makes a **live model call** —
every response below is real output from `groq/llama-3.3-70b-versatile` via
litellm, not scripted. Requires `GROQ_API_KEY` in the environment.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent.parent / "src"))
sys.path.insert(0, str(Path.cwd().parent / "code"))

from backbone_agent import run_agent
from backbone_agent.loop import SYSTEM_PROMPT

print(SYSTEM_PROMPT)

You are a coding agent. Solve the task by writing and running Python code.

On each turn:
- If you need to compute, inspect, or produce something, respond with a single ```python code block. It will be executed, and you will see its stdout (or an error traceback) as the next message.
- When the task is fully solved, respond with plain text and NO code block, stating the final answer. This ends the task.

Rules:
- Exactly one code block per turn — put all the logic for this step in it.
- Use print(...) for anything you need to see back; only printed output becomes visible to you.
- If your code raises an error, read the traceback in the next message, fix the code, and try again.



## 1. Smallest possible run

One trivial task, full trace shown, to see the loop's mechanics before the
three hands-on tasks: the model emits a code block, `run_agent` really
executes it, the real result is appended as the next message, and the model
is called again — this time producing plain text (no code block), which
`run_agent` recognizes as the final answer per the stop signal in
`SYSTEM_PROMPT`.

In [2]:
answer, messages = run_agent("What is 17 * 23? Compute it, don't just guess.", return_trace=True)
for m in messages:
    if m["role"] == "system":
        continue
    print(f"--- {m['role']} ---")
    print(m["content"].strip())
    print()
print("FINAL ANSWER:", answer)

--- user ---
What is 17 * 23? Compute it, don't just guess.

--- assistant ---
```python
result = 17 * 23
print(result)
```

--- user ---
Observation:
391

--- assistant ---
The result of 17 * 23 is indeed 391.

FINAL ANSWER: The result of 17 * 23 is indeed 391.


## 2. Three hands-on tasks

`three_tasks_demo.py` defines a math problem, a file transform (real CSV in,
real file written out), and an API-free data task, and verifies each against
an independently computed ground truth — not by trusting the agent's own
claim.

In [3]:
from three_tasks_demo import task_1_math, task_2_file_transform, task_3_data_stats, render_trace

results = []

### Task 1 — math (sum of the first 20 primes)

In [4]:
r1 = task_1_math()
results.append(r1)
print(render_trace(r1["messages"]))
print(f"\nExpected: {r1['expected']}  |  Success: {r1['success']}")

--- USER/OBSERVATION ---
Compute the sum of the first 20 prime numbers. State the final numeric answer clearly.
--- ASSISTANT ---
```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def sum_of_primes(n):
    count = 0
    num = 2
    total = 0
    while count < n:
        if is_prime(num):
            total += num
            count += 1
        num += 1
    return total

print(sum_of_primes(20))
```
--- USER/OBSERVATION ---
Observation:
639
--- ASSISTANT ---
The final answer is: 639

Expected: 639  |  Success: True


### Task 2 — file transform (average a CSV column, write a real file)

Watch this one closely: the model's first attempt commonly reaches for
`pandas`, which isn't installed in this minimal environment — a real
`ModuleNotFoundError`, not staged. Per the chapter's "first thing that will
go wrong" fill-in pointer, this *is* that first thing, caught live. The
traceback becomes the next Observation, and the model's second code action
switches to the stdlib `csv` + `statistics` modules and succeeds.

In [5]:
r2 = task_2_file_transform()
results.append(r2)
print(render_trace(r2["messages"]))
print(f"\nExpected: {r2['expected']}  |  Success: {r2['success']}")
print(f"File written: {r2['file_written']}  |  File contents: {r2['file_contents']!r}")

--- USER/OBSERVATION ---
Read the CSV file at /Users/bhakti/workspace/agentic/agent-code-execution/chapters/ch05-minimal-code-executing-agent/code/workspace/scores.csv (columns: name, score). Compute the average of the 'score' column, rounded to 2 decimal places, and write ONLY that number as text to /Users/bhakti/workspace/agentic/agent-code-execution/chapters/ch05-minimal-code-executing-agent/code/workspace/average.txt. Then state the final average in your answer.
--- ASSISTANT ---
```python
import pandas as pd

# Read the CSV file
df = pd.read_csv('/Users/bhakti/workspace/agentic/agent-code-execution/chapters/ch05-minimal-code-executing-agent/code/workspace/scores.csv')

# Compute the average of the 'score' column
average = round(df['score'].mean(), 2)

# Write the average to a file
with open('/Users/bhakti/workspace/agentic/agent-code-execution/chapters/ch05-minimal-code-executing-agent/code/workspace/average.txt', 'w') as f:
    f.write(str(average))

print(average)
```
--- USER/O

### Task 3 — API-free data task (mean/median/stdev)

In [6]:
r3 = task_3_data_stats()
results.append(r3)
print(render_trace(r3["messages"]))
print(f"\nExpected: {r3['expected']}  |  Success: {r3['success']}")

--- USER/OBSERVATION ---
Given the list [12, 45, 7, 22, 9, 34, 18], compute the mean, median, and population standard deviation. Round each to 2 decimal places and state all three clearly in your final answer.
--- ASSISTANT ---
```python
import numpy as np

# Given list
numbers = [12, 45, 7, 22, 9, 34, 18]

# Calculate mean
mean = round(np.mean(numbers), 2)
print(f"Mean: {mean}")

# Calculate median
median = round(np.median(numbers), 2)
print(f"Median: {median}")

# Calculate population standard deviation
std_dev = round(np.std(numbers, ddof=0), 2)
print(f"Population Standard Deviation: {std_dev}")
```
--- USER/OBSERVATION ---
Observation:
Traceback (most recent call last):
  File "/Users/bhakti/workspace/agentic/agent-code-execution/src/backbone_agent/executor.py", line 21, in execute_code
    exec(code, namespace)
    ~~~~^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'numpy'
--- ASSISTANT ---
```python
import statistics

# Given list
nu

## 3. Summary

In [7]:
n_success = sum(1 for r in results if r["success"])
print(f"{n_success}/{len(results)} tasks solved correctly")
for r in results:
    print(f"  - {r['name']}: {'OK' if r['success'] else 'FAILED'}")

3/3 tasks solved correctly
  - math (sum of first 20 primes): OK
  - file transform (average a CSV column): OK
  - data task (mean/median/stdev of an inline list): OK


Whatever `ModuleNotFoundError` (or other failure) appears above is real,
live-model output from whichever run actually happened when this notebook
was last executed — not guaranteed to be identical every time it's re-run
(the model can choose a different first attempt), which is itself the
chapter's point about the first thing that goes wrong: it's *emergent* from
a real, imperfect environment, not something the harness pre-declared.